# QMV Index: Interpretability

Global and local explanation of the QMV Index models.


In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import percentileofscore
from src.models.metrics import calculate_metrics


## Setup: Train Models

In [ ]:
df = pd.read_parquet('data/processed/features_t3.parquet').sort_values('time')
DROP = {'id','title','url','by','time','type','target','domain'}
fc = [c for c in df.columns if c not in DROP]

split = int(len(df)*0.7)
train, test = df.iloc[:split].copy(), df.iloc[split:].copy()
Xtr = np.ascontiguousarray(train[fc].values, dtype=np.float64)
ytr = np.ascontiguousarray(train['target'].values, dtype=np.float64)
Xte = np.ascontiguousarray(test[fc].values, dtype=np.float64)
yte = test['target'].values

mc = [1 if c in {'v1','v2','v3','acceleration','author_hist_median','domain_hist_median'} else 0 for c in fc]
m_tree = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=5, monotonic_cst=mc, random_state=42)
m_tree.fit(Xtr, ytr)

m_ridge = Pipeline([('sc', StandardScaler()), ('r', Ridge(alpha=10))])
m_ridge.fit(Xtr, ytr)

test = test.copy()
test['pred_tree'] = m_tree.predict(Xte)
test['pred_ridge'] = m_ridge.predict(Xte)
test['abs_err'] = (test['target'] - test['pred_tree']).abs()

print(f'Tree  RMSE: {calculate_metrics(yte, test["pred_tree"].values)["RMSE"]:.4f}')
print(f'Ridge RMSE: {calculate_metrics(yte, test["pred_ridge"].values)["RMSE"]:.4f}')


## 1. Global: Ridge Coefficient Plot (GLM Partial Effects)

In [ ]:
coef = pd.Series(m_ridge.named_steps['r'].coef_, index=fc).sort_values()
top = pd.concat([coef.head(10), coef.tail(10)])

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in top.values]
top.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Ridge Coefficients — Top 10 Positive and Negative Drivers')
ax.set_xlabel('Coefficient (on standardized features)')
plt.tight_layout()
plt.show()


## 1.2 Global: Tree Feature Importances

In [ ]:
fi = pd.Series(m_tree.feature_importances_, index=fc).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
fi.sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_title('HistGBT Feature Importances — Top 15')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

# Model agreement check
ridge_top = set(pd.Series(m_ridge.named_steps['r'].coef_, index=fc).abs().sort_values(ascending=False).head(8).index)
tree_top = set(fi.head(8).index)
overlap = ridge_top & tree_top
print(f'Agreement: {len(overlap)}/8 features in common between Ridge top-8 and Tree top-8')
print('Shared:', sorted(overlap))


## 1.3 GAM-style Partial Dependence Plots (sklearn)

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

top_feats = ['v3', 'v1', 'acceleration', 'author_hist_median', 'market_top30_vel']
feat_idx = [fc.index(f) for f in top_feats if f in fc]

fig, axes = plt.subplots(1, len(feat_idx), figsize=(16, 4))
PartialDependenceDisplay.from_estimator(
    m_tree, Xtr, feat_idx, feature_names=fc,
    ax=axes, grid_resolution=50
)
fig.suptitle('Partial Dependence Plots — HistGBT (monotonic constraints enforced)')
plt.tight_layout()
plt.show()


## 2. Local: explain_story()

In [ ]:
def explain_story(story_id, df, m_tree, m_ridge, fc, T=3):
    """Returns QMV score and plain-English attribution for a story."""
    mask = df['id'] == story_id
    row = df[mask].iloc[0] if mask.any() else df.iloc[story_id % len(df)]
    
    Xrow = np.ascontiguousarray(row[fc].values.reshape(1, -1), dtype=np.float64)
    pred_val = m_tree.predict(Xrow)[0]
    
    all_preds = m_tree.predict(np.ascontiguousarray(df[fc].values, dtype=np.float64))
    qmv = percentileofscore(all_preds, pred_val, kind='weak')
    
    # Attribution via Ridge
    coef = m_ridge.named_steps['r'].coef_
    sc = m_ridge.named_steps['sc']
    z = (row[fc].values - sc.mean_) / (sc.scale_ + 1e-8)
    contribs = {feat: float(coef[i] * z[i]) for i, feat in enumerate(fc)}
    top5 = sorted(contribs.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
    
    parts = []
    for feat, cont in top5:
        sign = '+' if cont >= 0 else ''
        label = feat.replace('_', ' ')
        parts.append(f'{sign}{cont:.1f} from {label}')
    
    return {
        'story_id': story_id,
        'title': str(row.get('title', 'N/A')),
        'T_hours': T,
        'qmv_index': round(qmv, 1),
        'predicted_log_value': round(pred_val, 3),
        'actual_log_value': round(float(row.get('target', 0)), 3),
        'attribution': '; '.join(parts),
    }

# Demo
ex = explain_story(test.iloc[0]['id'], test, m_tree, m_ridge, fc)
for k, v in ex.items():
    print(f'{k:25s}: {v}')


## 3. Five Worked Examples

In [ ]:
high_correct = test.sort_values('pred_tree', ascending=False).iloc[0]
low_correct  = test[test['pred_tree'] < test['target'].quantile(0.1)].sort_values('abs_err').iloc[0]
big_miss     = test.sort_values('abs_err', ascending=False).iloc[0]

sb_mask = (test['v1'] < test['v1'].quantile(0.2)) & (test['target'] > test['target'].quantile(0.7))
slow_burn = test[sb_mask].iloc[0] if sb_mask.any() else test.iloc[5]

ff_mask = (test['v1'] > test['v1'].quantile(0.8)) & (test['target'] < test['target'].quantile(0.3))
fast_fade = test[ff_mask].iloc[0] if ff_mask.any() else test.iloc[10]

cases = {
    '🎯 Correct High': high_correct,
    '✅ Correct Low': low_correct,
    '❌ Big Miss': big_miss,
    '🐢 Slow Burner': slow_burn,
    '🔥 Fast Fader': fast_fade,
}

for label, row in cases.items():
    ex = explain_story(row['id'], test, m_tree, m_ridge, fc)
    print(f'--- {label} ---')
    print(f'  Title            : {ex["title"]}')
    print(f'  QMV Index        : {ex["qmv_index"]}')
    print(f'  Predicted / Actual: {ex["predicted_log_value"]} / {ex["actual_log_value"]}')
    print(f'  Why              : {ex["attribution"]}')
    print()


## 4. Sanity Checks

In [ ]:
coef_s = pd.Series(m_ridge.named_steps['r'].coef_, index=fc)
checks = {
    'v3 positive?':                coef_s.get('v3', 0) > 0,
    'acceleration positive?':      coef_s.get('acceleration', 0) > 0,
    'is_weekend negative?':        coef_s.get('is_weekend', 0) < 0,
    'author_hist_median positive?': coef_s.get('author_hist_median', 0) > 0,
    'competing_stories negative?': coef_s.get('competing_stories_hr', 0) < 0,
}
for check, result in checks.items():
    emoji = '✅' if result else '❌ UNEXPECTED'
    print(f'{emoji}  {check}')

print()
print('⚠️  Fairness note: author_hist_median acts as an author-identity proxy.')
print('   New authors will be penalised regardless of content quality.')
print('   Mitigation: cap to 90-day window, add is_new_author flag.')


## 5. Why a Buyer or Seller Should Trust This Number

See `reports/interpretability.md` for the full written version. Key points:

1. **No forward-looking data** — every feature is observable at prediction time.
2. **Monotonic constraints** — the model cannot predict a *lower* value for a *more* popular story.
3. **Signed attributions** — every QMV score comes with a breakdown of what drove it up or down.
4. **Rolling backtest with drift monitoring** — model quality is tracked fold-by-fold over time.
5. **Honest failure analysis** — the 20 worst errors are published and explained, not hidden.
